# 07 


Phase 1: run one-audio temporal explanations

Run a single explanation:

.venv/bin/python scripts/run_relevance_explanation.py \
  --audio path/to/audio.wav \
  --explanation-mode contrastive-conditioned-rollout \
  --device cpu \
  --local-files-only \
  --output-root outputs/test_04_single_audio/run_01_explanation

Available modes:

rollout
level3
level3-contrastive
head-conditioned-rollout
contrastive-conditioned-rollout

You can force a target class:

.venv/bin/python scripts/run_relevance_explanation.py \
  --audio path/to/audio.wav \
  --explanation-mode level3 \
  --target-class 2 \
  --device cpu \
  --local-files-only \
  --output-root outputs/test_04_single_audio/run_02_target_ang

For contrastive modes, you can also force the competitor class:

.venv/bin/python scripts/run_relevance_explanation.py \
  --audio path/to/audio.wav \
  --explanation-mode level3-contrastive \
  --target-class 2 \
  --contrast-class 3 \
  --device cpu \
  --local-files-only \
  --output-root outputs/test_04_single_audio/run_03_ang_vs_sad

Each run writes:

*_temporal.csv       # token/time relevance values
*_timeline.png       # waveform/spectrogram/relevance visualization
*_metadata.json      # source audio, class, mode, and output provenance

Phase 2: evaluate prediction performance

These scripts produce the prediction CSVs used by the faithfulness and class-specificity evaluations.
RAVDESS smoke test

.venv/bin/python scripts/evaluate_ravdess.py \
  --max-samples 60 \
  --batch-size 1 \
  --device cpu \
  --local-files-only \
  --output-root outputs/test_02_ravdess_explanation_benchmark/run_02_external_smoke

RAVDESS full external evaluation

.venv/bin/python scripts/evaluate_ravdess.py \
  --batch-size 1 \
  --device cpu \
  --local-files-only \
  --output-root outputs/test_02_ravdess_explanation_benchmark/run_03_external_full

IEMOCAP smoke test

.venv/bin/python scripts/evaluate_iemocap.py \
  --max-samples 4 \
  --batch-size 1 \
  --device cpu \
  --local-files-only \
  --output-root outputs/test_03_iemocap_in_domain_benchmark/run_01_evaluation_smoke

IEMOCAP full in-domain diagnostic

.venv/bin/python scripts/evaluate_iemocap.py \
  --batch-size 1 \
  --device cpu \
  --local-files-only \
  --output-root outputs/test_03_iemocap_in_domain_benchmark/run_04_evaluation_full

IEMOCAP note: the public checkpoint is already fine-tuned for IEMOCAP-style emotion recognition. Treat IEMOCAP results as in-domain diagnostics unless you define a separate held-out protocol.

Prediction evaluation outputs:

predictions.csv
metrics.json
confusion_matrix.csv
run_manifest.json

Phase 3: deletion-faithfulness evaluation

Deletion faithfulness tests whether removing the most relevant time tokens reduces the target class logit/probability more than removing bottom-k or random tokens.

Use a predictions.csv produced by Phase 2.

Example for IEMOCAP:

.venv/bin/python scripts/evaluate_deletion_faithfulness.py \
  --predictions-csv outputs/test_03_iemocap_in_domain_benchmark/run_04_evaluation_full/iemocap_in_domain_eval_TIMESTAMP/predictions.csv \
  --dataset-name IEMOCAP \
  --max-examples 8 \
  --fractions 0.05,0.1,0.2,0.3 \
  --random-trials 3 \
  --device cpu \
  --local-files-only \
  --output-root outputs/test_03_iemocap_in_domain_benchmark/run_05_faithfulness_full

Example for RAVDESS:

.venv/bin/python scripts/evaluate_deletion_faithfulness.py \
  --predictions-csv outputs/test_02_ravdess_explanation_benchmark/run_03_external_full/ravdess_external_eval_TIMESTAMP/predictions.csv \
  --dataset-name RAVDESS \
  --max-examples 8 \
  --fractions 0.05,0.1,0.2,0.3 \
  --random-trials 3 \
  --device cpu \
  --local-files-only \
  --output-root outputs/test_02_ravdess_explanation_benchmark/run_05_faithfulness_all_modes

Deletion outputs:

audio_manifest.csv
selected_examples.csv
class_coverage.csv
deletion_records.csv
deletion_summary.csv
deletion_curves.png
sparsity_records.csv
sparsity_summary_by_method.csv
sparsity_summary_by_class.csv
robustness_by_class.csv
config.json

Key deletion interpretation:

    Stronger faithfulness: top-k deletion should decrease the target probability or logit more than bottom-k and random deletion.
    Negative deletion / bottom-k is a sanity check: deleting the least relevant regions should usually have a smaller effect.
    robustness_by_class.csv helps check whether behavior is stable across emotional classes.

Phase 4: sparsity / concentration metrics

Sparsity is already computed inside the deletion-faithfulness script. The main summary file is:

sparsity_summary_by_method.csv

It reports:

normalized_entropy
effective_tokens
gini
top_5_percent_mass
top_10_percent_mass

Interpretation:

    Higher normalized entropy = more diffuse explanation.
    Higher effective tokens = relevance spread across more time tokens.
    Higher Gini = more concentrated explanation.
    Higher top-5/top-10 mass = more relevance concentrated in the most relevant time tokens.

This table is useful for formalizing:

rollout is diffuse
contrastive variants are more selective

Phase 5: class-specificity evaluation

Class-specificity compares the relevance map for the predicted class with the map for the runner-up class on the same audio.

Lower correlation means the method is more class-specific.

Run the batch metric:

.venv/bin/python scripts/evaluate_class_specificity.py \
  --predictions-csv outputs/test_03_iemocap_in_domain_benchmark/run_04_evaluation_full/iemocap_in_domain_eval_TIMESTAMP/predictions.csv \
  --dataset-name IEMOCAP \
  --max-examples 8 \
  --device cpu \
  --local-files-only \
  --output-root outputs/test_03_iemocap_in_domain_benchmark/run_09_class_specificity_metrics_full

Outputs:

audio_manifest.csv
selected_examples.csv
class_coverage.csv
class_specificity_records.csv
class_specificity_summary_by_method.csv
class_specificity_summary_by_class.csv
config.json

The main columns are:

pearson_correlation
spearman_correlation

Interpretation:

    High positive correlation: predicted-class and runner-up maps are similar.
    Low or negative correlation: maps differ more strongly by target class.
    A contrastive method is expected to have lower correlation than plain rollout.
